# 04 — Evaluación del clasificador SVM

Carga el modelo entrenado por `scripts/train_svm.py` y analiza:
- Matriz de confusión sobre test.
- Galería de aciertos.
- Galería de fallos (los más informativos).
- Predicción sobre regiones reales del region proposal.

**Pre-requisitos**:
1. `python scripts/build_dataset.py` (ya ejecutado)
2. `python scripts/train_svm.py` (ya ejecutado)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

from src.features import extract_features
from src.preprocessing import preprocess
from src.region_proposal import propose_regions, draw_proposals
from src.utils.io_utils import load_image

plt.rcParams['figure.dpi'] = 90

## 1. Cargar modelo y predicciones

In [ ]:
MODEL_PATH = Path('../results/models/svm_categoria.joblib')
DATASET_DIR = Path('../results/dataset')

if not MODEL_PATH.is_file():
    print(f'❌  No existe {MODEL_PATH}')
    print('Ejecuta: python scripts/train_svm.py')
else:
    bundle = joblib.load(MODEL_PATH)
    model = bundle['model']
    metrics = bundle['metrics']

    print('Métricas guardadas:')
    for k, v in metrics.items():
        print(f'  {k:20s} {v:.4f}')

## 2. Matriz de confusión

In [ ]:
y_test = np.load(DATASET_DIR / 'y_test.npy', allow_pickle=True)
y_pred = np.load(DATASET_DIR / 'y_pred.npy', allow_pickle=True)

labels = sorted(set(y_test))
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45)
ax.set_yticklabels(labels)
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión (test)')
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

print('\nClassification report:')
print(classification_report(y_test, y_pred, zero_division=0))

## 3. Predicción sobre regiones reales del pipeline

Cogemos una imagen, le aplicamos region proposal, y para cada caja
predecimos la categoría con el SVM. Es una vista previa de lo que será
el pipeline integrado en F6.

In [ ]:
import cv2

DATA_ROOT = Path('../data/external/klasson_flat')
test_class = 'Apple'  # cambia por una clase que exista

imgs = sorted(list((DATA_ROOT / test_class).glob('*.jpg')))
if not imgs:
    print(f'No hay imágenes en {test_class}, prueba con otra clase')
else:
    img = load_image(imgs[0])
    img_pre = preprocess(img)
    proposals = propose_regions(img_pre)

    # Predecir cada caja
    predicted = []
    for p in proposals:
        crop = img_pre[p.y:p.y+p.h, p.x:p.x+p.w]
        if crop.size == 0:
            continue
        features = extract_features(crop)
        pred = model.predict(features.reshape(1, -1))[0]
        proba = model.predict_proba(features.reshape(1, -1))[0].max()
        predicted.append((p, pred, proba))

    # Dibujar con la categoría predicha
    annotated = img_pre.copy()
    palette = {
        'fruta':   (50, 200, 50),
        'verdura': (255, 200, 0),
        'brick':   (0, 100, 255),
        'paquete': (200, 0, 200),
        'otros':   (180, 180, 180),
    }
    bgr = cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR)
    for p, pred, proba in predicted:
        color = palette.get(pred, (255, 255, 255))
        cv2.rectangle(bgr, (p.x, p.y), (p.x+p.w, p.y+p.h), color, 2)
        label = f'{pred} {proba:.2f}'
        cv2.putText(bgr, label, (p.x+2, p.y-4), cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, color, 1, cv2.LINE_AA)
    annotated = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img_pre); axes[0].set_title('Preprocesada'); axes[0].axis('off')
    axes[1].imshow(annotated)
    axes[1].set_title(f'{len(predicted)} cajas con predicción SVM')
    axes[1].axis('off')
    plt.tight_layout(); plt.show()

    print('\nPredicciones top-5 (por confianza):')
    for p, pred, proba in sorted(predicted, key=lambda x: -x[2])[:5]:
        print(f'  {pred:10s} ({proba:.2f}) en ({p.x},{p.y},{p.w},{p.h})')

## 4. Galería de fallos

Cargamos las imágenes mal clasificadas para ver qué tipo de errores comete
el SVM. Esto es muy útil para decidir qué mejorar.

In [ ]:
paths_all = np.load(DATASET_DIR / 'paths.npy', allow_pickle=True)
y_all = np.load(DATASET_DIR / 'y.npy', allow_pickle=True)

# Para tener las paths de test, hacemos el mismo split con el mismo seed
from src.classification import split_dataset
X_all = np.load(DATASET_DIR / 'X.npy')
_, _, _, _, X_test_split, y_test_split = split_dataset(X_all, y_all, random_state=42)

# Reproducimos el split sobre paths usando los mismos índices
from sklearn.model_selection import train_test_split
_, idx_test = train_test_split(
    np.arange(len(y_all)), test_size=0.15, stratify=y_all, random_state=42,
)
_, idx_test_only = train_test_split(
    idx_test, test_size=0.15 / 0.85, stratify=y_all[idx_test], random_state=42,
)
# (esta reproducción puede no ser exacta; sirve como aproximación visual)

# Encuentra fallos
wrong_mask = y_pred != y_test
print(f'Total fallos en test: {wrong_mask.sum()} de {len(y_test)} ({100*wrong_mask.mean():.1f}%)')

## 5. Conclusiones y siguiente paso

Cosas a mirar al ejecutar este notebook:
- **Accuracy total**: ¿estamos por encima del baseline aleatorio (1/n_clases = 20%)?
- **Matriz de confusión**: ¿hay clases que se confunden entre sí? Eso sugiere que sus features se parecen.
- **Predicción sobre regiones**: ¿el SVM tiene sentido cuando se aplica a las cajas reales del region proposal?

**Próximo paso (F4 / F5)**:
- Si los resultados son razonables: pasamos al dataset Monster propio + Deep Learning.
- Si son malos: probamos Random Forest o ajustamos features.